In [4]:
# -- Cora Dataset --
import time
import random
from torch_geometric.datasets import Planetoid
import torch_geometric.utils as utils
import networkx as nx
from sklearn.metrics import roc_auc_score, average_precision_score

dataset = Planetoid(root='./data/Cora', name="Cora")
data = dataset[0]
test_ratio = 0.1

# 전체 그래프 networkx graph
g_nx = utils.to_networkx(data, to_undirected=True)
all_edges = list(g_nx.edges())
num_test_edges = int(len(all_edges) * test_ratio) # 


# 테스트용 양성 샘플 선택 후 그래프에서 제거 (학습용 그래프 생성)
random.seed(42)
test_edges_pos = random.sample(all_edges, num_test_edges)
train_g = g_nx.copy()
train_g.remove_edges_from(test_edges_pos)

# 음성 샘플 샘플링(=pos)
test_edges_neg = []
nodes = list(g_nx.nodes())
while len(test_edges_neg) < num_test_edges:
    u, v = random.sample(nodes, 2)
    if u != v and not g_nx.has_edge(u, v):
        test_edges_neg.append((u, v))

def get_pagerank_scores(graph, edge_list):
    scores = []
    
    # 각 노드별 PPR 결과를 캐싱하여 반복 계산을 줄입니다.
    # 대규모 그래프에서는 모든 노드를 미리 계산하는 것이 메모리/시간 면에서 효율적일 수 있습니다.
    ppr_cache = {}
    
    for u, v in edge_list:
        # 노드 u에서 시작하는 Personalized PageRank 계산
        if u not in ppr_cache:
            # alpha는 덤핑 팩터(Damping Factor)로, 보통 0.85를 사용합니다.
            # 빌트인 nx.pagerank에서 personalization 딕셔너리를 주면 PPR이 됩니다.
            ppr_cache[u] = nx.pagerank(graph, alpha=0.85, personalization={u: 1})
        
        # u에서 출발해 v에 도달할 확률 점수 저장
        scores.append(ppr_cache[u].get(v, 0.0))
    
    return scores

start_time = time.time()
pos_scores = get_pagerank_scores(train_g, test_edges_pos)
neg_scores = get_pagerank_scores(train_g, test_edges_neg)

y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
y_scores = pos_scores + neg_scores

auc = roc_auc_score(y_true, y_scores)
ap = average_precision_score(y_true, y_scores)
print(f'--test pos edge number: {num_test_edges}, test neg edge number: {len(test_edges_neg)}')
print(f"Planetoid Cora 데이터셋 기준 Pagerank 링크 예측 AUC: {auc:.4f}, AP: {ap:.4f}")
print(f"걸린 시간: {time.time() - start_time:.4f} (s)")



--test pos edge number: 527, test neg edge number: 527
Planetoid Cora 데이터셋 기준 Pagerank 링크 예측 AUC: 0.8560, AP: 0.9000
걸린 시간: 8.4718 (s)


In [5]:
# -- Citeseer Dataset --
import time
import random
from torch_geometric.datasets import Planetoid
import torch_geometric.utils as utils
import networkx as nx
from sklearn.metrics import roc_auc_score, average_precision_score

dataset = Planetoid(root='./data/Citeseer', name="Citeseer")
data = dataset[0]
test_ratio = 0.1

# 전체 그래프 networkx graph
g_nx = utils.to_networkx(data, to_undirected=True)
all_edges = list(g_nx.edges())
num_test_edges = int(len(all_edges) * test_ratio) # 


# 테스트용 양성 샘플 선택 후 그래프에서 제거 (학습용 그래프 생성)
random.seed(42)
test_edges_pos = random.sample(all_edges, num_test_edges)
train_g = g_nx.copy()
train_g.remove_edges_from(test_edges_pos)

# 음성 샘플 샘플링(=pos)
test_edges_neg = []
nodes = list(g_nx.nodes())
while len(test_edges_neg) < num_test_edges:
    u, v = random.sample(nodes, 2)
    if u != v and not g_nx.has_edge(u, v):
        test_edges_neg.append((u, v))

def get_pagerank_scores(graph, edge_list):
    scores = []
    
    # 각 노드별 PPR 결과를 캐싱하여 반복 계산을 줄입니다.
    # 대규모 그래프에서는 모든 노드를 미리 계산하는 것이 메모리/시간 면에서 효율적일 수 있습니다.
    ppr_cache = {}
    
    for u, v in edge_list:
        # 노드 u에서 시작하는 Personalized PageRank 계산
        if u not in ppr_cache:
            # alpha는 덤핑 팩터(Damping Factor)로, 보통 0.85를 사용합니다.
            # 빌트인 nx.pagerank에서 personalization 딕셔너리를 주면 PPR이 됩니다.
            ppr_cache[u] = nx.pagerank(graph, alpha=0.85, personalization={u: 1})
        
        # u에서 출발해 v에 도달할 확률 점수 저장
        scores.append(ppr_cache[u].get(v, 0.0))
    
    return scores

start_time = time.time()
pos_scores = get_pagerank_scores(train_g, test_edges_pos)
neg_scores = get_pagerank_scores(train_g, test_edges_neg)

y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
y_scores = pos_scores + neg_scores

auc = roc_auc_score(y_true, y_scores)
ap = average_precision_score(y_true, y_scores)
print(f'--test pos edge number: {num_test_edges}, test neg edge number: {len(test_edges_neg)}')
print(f"Planetoid Citeseer 데이터셋 기준 Pagerank 링크 예측 AUC: {auc:.4f}, AP: {ap:.4f}")
print(f"걸린 시간: {time.time() - start_time:.4f} (s)")



--test pos edge number: 455, test neg edge number: 455
Planetoid Citeseer 데이터셋 기준 Pagerank 링크 예측 AUC: 0.7256, AP: 0.8315
걸린 시간: 7.6797 (s)


In [ ]:
# -- Pubmed Dataset --
import time
import random
from torch_geometric.datasets import Planetoid
import torch_geometric.utils as utils
import networkx as nx
from sklearn.metrics import roc_auc_score, average_precision_score

dataset = Planetoid(root='./data/Pubmed', name="Pubmed")
data = dataset[0]
test_ratio = 0.1

# 전체 그래프 networkx graph
g_nx = utils.to_networkx(data, to_undirected=True)
all_edges = list(g_nx.edges())
num_test_edges = int(len(all_edges) * test_ratio) # 


# 테스트용 양성 샘플 선택 후 그래프에서 제거 (학습용 그래프 생성)
random.seed(42)
test_edges_pos = random.sample(all_edges, num_test_edges)
train_g = g_nx.copy()
train_g.remove_edges_from(test_edges_pos)

# 음성 샘플 샘플링(=pos)
test_edges_neg = []
nodes = list(g_nx.nodes())
while len(test_edges_neg) < num_test_edges:
    u, v = random.sample(nodes, 2)
    if u != v and not g_nx.has_edge(u, v):
        test_edges_neg.append((u, v))

def get_pagerank_scores(graph, edge_list):
    scores = []
    
    # 각 노드별 PPR 결과를 캐싱하여 반복 계산을 줄입니다.
    # 대규모 그래프에서는 모든 노드를 미리 계산하는 것이 메모리/시간 면에서 효율적일 수 있습니다.
    ppr_cache = {}
    
    for u, v in edge_list:
        # 노드 u에서 시작하는 Personalized PageRank 계산
        if u not in ppr_cache:
            # alpha는 덤핑 팩터(Damping Factor)로, 보통 0.85를 사용합니다.
            # 빌트인 nx.pagerank에서 personalization 딕셔너리를 주면 PPR이 됩니다.
            ppr_cache[u] = nx.pagerank(graph, alpha=0.85, personalization={u: 1})
        
        # u에서 출발해 v에 도달할 확률 점수 저장
        scores.append(ppr_cache[u].get(v, 0.0))
    
    return scores

start_time = time.time()
pos_scores = get_pagerank_scores(train_g, test_edges_pos)
neg_scores = get_pagerank_scores(train_g, test_edges_neg)

y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
y_scores = pos_scores + neg_scores

auc = roc_auc_score(y_true, y_scores)
ap = average_precision_score(y_true, y_scores)
print(f'--test pos edge number: {num_test_edges}, test neg edge number: {len(test_edges_neg)}')
print(f"Planetoid Pubmed 데이터셋 기준 Pagerank 링크 예측 AUC: {auc:.4f}, AP: {ap:.4f}")
print(f"걸린 시간: {time.time() - start_time:.4f} (s)")

